# AeroVision Lab — Kaggle resolution evaluation
Keep this notebook and its input Dataset **private**. Upload the completed
`colab-full-v1-60epochs-b16-results.zip` as a private Kaggle Dataset and attach it.
Enable Internet and an available GPU accelerator. This notebook does not train.
It uses one GPU even if T4 x2 is assigned, keeping inference comparable to Colab.
Download the final results ZIP before ending the session. No Google Drive or
GitHub token is needed; the source snapshot is embedded below.


In [ ]:
import os, sys, pathlib, subprocess, json, csv, zipfile, io, base64, hashlib
assert pathlib.Path('/kaggle/working').exists(), 'Run this notebook on Kaggle.'
PROJECT = pathlib.Path('/kaggle/working/aerovision-lab')
PROJECT.mkdir(exist_ok=True)
os.chdir(PROJECT)
os.environ['YOLO_CONFIG_DIR'] = str(PROJECT / '.state/ultralytics')
(PROJECT / '.state/ultralytics').mkdir(parents=True, exist_ok=True)


In [ ]:
bundle = base64.b64decode('UEsDBBQAAAAIAAAAIQB4tUodYAEAAFECAAAOAAAAcHlwcm9qZWN0LnRvbWxVUctqwzAQvOsrhM6xsB0IaakMPfXSQqD0UIwIsryO1ciSKslJ/fdFdsjjOLuzO7OzdTMq3WZhChEGjjz8jspDwAzXJEAcXbRWh4pttoSjhdsIeQTTYobvGHTu7QeIgiBUO29/QEaOjBggMQV4e1JBWZNp0RB0Ap9AauW0oDlBLQTplYuX6ucgtMZvuy8Mfw68GsDEgDvrsQCvhMYhETLbJB3cQgSZRsn1hMxNsV92VWxNiyJJODAtGKkuF446eqGnqGRgbEvXtCy3ZIWJGQc3Vayg5Wb1sk6VndLanitW5DOavl8/3iu2ScBN0kp7yamkOU1RXSOgdj5J6OxenaMWTrMFN0UIsWKzrh+7rmI5fXrYsOQSOLqF+BgplVo9D0KZFH0yQu8e44Q8igME2inTcnTuwaeP1CR4OcvMA4sNqozaL4YDR6niROyXrBIKt4FklSOtDGQazCH2mOGiyNE/UEsDBBQAAAAIAAAAIQDrzMOnIQwAAG8aAAAJAAAAUkVBRE1FLm1kfVltb9w2Ev4uQP9hgH64xtjVrt3E18a9AobTpkaT2EjiFIeiOHOpkcQzxVFJatfbX3+Yod7ipvclSHa55Lw888wzk6/gEj19MsGQgzdql2d59qFV1q5p91/UEUqMqCN/i48detOiiwHIwScTXnly+BIwRLWzJjSgwGPnqey12VmEnQpojcMVxAYdGLfHEE2tIoLHQLbne1cQjcUSjKvQo9MIypVQYmfpyI8VbNLHxgQwARTfUmKHrkQXwbSdRT6kxMIQlY9YQuWphaC9irop4B1B9Mo442p+trcxQKP2CDtEB8lcLOGI6amvvoIPUdUIpy8nb5AdgKjCA584LeCTsqZkP/hzdHvjybEdYvvr2zs4mMjhCA35CKGlBwTfuyLPzgq49dgpj1ME02F87KzRJoK2KgRoVdcZV68gdNZEaJUzFYbIIShBOUeDz7pB/RCKPPumgI/sJij4982bm9PTMMUfVITz51vozCPaUMAH9l4ysMcSAsZoXB1WMAZWH2GPniERVhAQyxU0ypcHNpqflzc7Mi5Co0KD/PzzAn7cK9tzUMjBPgWILSxVVBf8HPo9QsQQ1yXuoSIPlXHKAqbf8VlVRfQQ0DLmXJ3CO+GOkxVJky3y7EUBV9RKHM+fb1fw3flWbDs9+3YLxnV9XGAsjPnQ5KIny3ibMBEw9l0BVw1RQFCwY9hAMH9ydlWEynDUrQWd3is5kykPHjV5TscRStwb8WHAK0JlfIhQk7KM3EhfQJNcMpcP9R7o4KBFFXovwA4FvGV72FDlKDbon6D+HwHwUekIrm936KVMHEVQIfQtlp9j+uwl/9rTHvnj9RTCQ0MW16blM2wT7dHbhMC/lGcf+FNNLpgQOStBk0/2lTOKQ991HoPwyoiwgp+8XrAAek8+gEPl5RnYUe9K5Q2G1WhDooOer+WftOhr42q56j12qbwWfOVRK2tX0KFfp0q6vF1BpWxA6CiYaPZ8eYst+aPcbVUUyCtLrg6mRHmZE95e3so7n9Cb6si+kfBLSyVa6DyWRqgxgWGAMS64i62oyLfKaZRE/LhHfxxTICfaPkTY4QyukTkU1+ZUweQENEG1KAWVokK650uwXFaGpDtRATmoPCK8JqotwhXZxPAMT6u6SB2Epo8BSoZd70r0wl6WVLmCsCDORgVoiU2KBAoaCvzq69u7It1GtgRLWln5ve+dY6AG8BiNx/ICrt5cp9s2g6fg6AAlVqq3kS+9kruEXV/RwbEJ8NvcnP4jxhemO7rd7187irgjegibL594lpj2pkMHTYxdeLnZaPmaeUh53RS1xKTQ1G4GUpP6Pzn5iYH4A9x1YsL40slJ4tiBJk5O3vcumpZPXjXK1cLv8kE8dvzp69u7kxO5uRdaGUI2nEqM+b5PadVobQBOmOcUCMV3gpExda+82SMorTEEOHA37Ty1XRwKXPrjaCpgu8MygGIWN5Vhjqfea4TgVBcaipJbR/DaxJ/7HUR64PbM2fqj53QVefYLYgeGKdfsVcQCXqmoAnLPVTUG7rVHxpcEfWy2ECJ5VeMFuwnKR1MpnXpWns1dIwi6B7zdvz0m7xapfKN2GybZzT2g0g1gR9zJ3yMDa+gLeTb563HXG1uGuURCv2NbE0czD4p06F1lnAkNB0TKazapyLPrCMoGkq/SVYlXh9jtelda5CSJtQX8JJV1ezdmRR5L2c0zayrkvwXYK3+8gPhZgriamKOPyLWPrK1Q9wwP4+DIbSCFVWlN/SSBEDpjKTKewtDj3Qq+3W5TYcEGzrbbZedNqVotev9q6G5necbmnm1TaEMB1zHJp5DU09XN1c268yg3MyuhqRvuRmwGKm8N+qHgRd3kGZe1xci1zrFL3yWDD4rBHdH7nhELO6zI49A9pXPA9Osh1wONTZIiZZZzNlxpYkPS40eK4l8MNbukjeEGbsz/jzueHnsmlJT4kiOVJMnnzAdvOTkz6FKN9oE9PMrHYivHbJH8oRASwhZK6sPPl+uzF+cCoxl85E0tIin15pZL24fGdBuWMujiKs9EXA+sGhZtCaxpWbt8s92u4Hw76KPtdpt6jDyS3CafzkJlHrHMMxXlXAGvRP2TB+ojSyq+oba0U1YaMY5PMMghUo0sUAQAxrNSN3tT9srmGVYVChMI8UcwgbjvlglQ32y3ozRLNMSSAsuBIVLt4GG2luurNYnGxnbCPplEp3yWNRUWcGntJPo9tlwmS2lqVO0oRKNZm+TZG05g6HedJynpwWvDdOdRtan9yRNDNgv4iUO21uQqw9IZYW9Cz6qWtU2e4aNiZA+6va49ivKpPWJ5XLdSjVLkKTaBRwM+MFUhXN5Ci9EnG5OwZEDcSz9bC8TW+9P1rKLX+7PiT9Pdj0xGY/cQyZBaaxD97XFv8DAW2y+q5lYjZbZOGmeW0AsZzsevW5Fe918spnRRKqR7rnyWDEMjGV+ZA5iKNs8Gh6reWvbnfJuoab07PV8PKUxuKW5tJyfDhScn45Vjh5JIx6h0k2eGhXCaBwr40TEo4JqpyI0HHai9Mla+GdncomeYDXOrX3bpPBvbtEyWn7XnVviA/N/3VUH71AYSEUgBL5lABPhY90/pfJi4VotMLo6Qs8dVYqyJEJbafjFky0i4+e58u5GRKVHC00lcDhV5duMsd/sUIROE5BY67X7zICnYHMg/GFdvFHraCxjWM4Q+y2KeCcLZ9VRlm8pwmXTKoU0KZ+wJTwgUSsLUPgfyH/F7dXvHYxha6kaYfpw60XJIT0QQFonnYhj6PJYy77lS7uNxNbCn/GCeMRQ4BDt0ummVfwiSbj44KmVR6iYM+jplnG/BEhrD+siwOb8aV9Ih5NnSLkEK98QkBO5L0mEzYEeGwbX4UsTHeH8xSASPUXwxMeRZ59G4EJUMuLfHj+R1sxhUU88ZR3tQOvbK2mNqWcbxHQL2FqPiIWMM7Nt5Hp0GjTQ/snO8IFg/Pxs1Vxj4jqrKaKPsIE7YiiWW5y2OPa5GL/IMZNKO6CAqX+OwDWG8T4uHRF1MtlFCmZA9tENQ2lMIaWvChAkJPBNm5ml7NkEjUDXwtj1CMK2xykPQ6GS3sYabsRx5dOAJEmZRwLFLmli2SmMfT3XKYZ3l0LjFSYNw7aTJSblz3xxH2B098gMegVoTp4UWK74ZZik8IshkNSahS+14oIIq4XCngtHz7zS5AQAv2bAjaOV4/uT+xk8p/oV+qD1P4sW8IzCBXFKe3VGTpkhkZQ84yZ8hCEIkrXp8hTH8S6TH3BvZyG2x3Z6K0+/efoBruoNt8c9iiobHOoFTBCKvENKAq8JipNeeDmWK05xiXg8uR2KZoB5NK+FYSfKFbUZcTuu3SVaIu78+WYUkTlyoKh7nZvQ5jEx6f1k5pZkH5OepbMXfaRUCH5/eq8oyjOJK3Fwr2bi9e/uhgMs47kNSDM9WoJib16Lr0/JElXteNARQO5bGL07Pxo0f+/VmWHMYp21fJqE46BzZMwqzj8y/Gvcs0wjFQ7pXLlToV7DrI5uBj8NVEq01EziUqIk1fZHWoBKaAOHodOPJmT8HwMtYxxoDDsqzruo7yeQH/KPn0DJv8HVzL6rUnnofwNJh3N7wDqpvuwT3PXroUD1AbDz1dcMdn/3m9c/l7Yvt6vM90eXt5kurIjGuxdIot/nuRWzWHXrNBlnemEgEpdJYcHouTpcAv1gGrcZhSShgNfIi/yOx4MC/Yt3HBnk74GXNaUlkx3FJlEPPvOBxfdqd8vwoc+MoApAVQ+rKTMzveUEpM/Zy6EZZPg2z9EfeR2qJmizV5+nM944JmVsVji0hz0rjZRJIOzOugftE7Ou17KO+P6rW/gDrtWMu+J7/5H+l6Ru+71RsNpE2VoVYdPGH+yLP3uEhrVI9SkoUi0wTZPHLbWh+kgXknkwJwVjpGJJuTsBiqT9tOqWPDwytqZNV5/Xf/q/BRWI1poY0Uw/rPan9kTXR1byFYx7Os9/ubPTKHnls+P3rcb3Ezbro529ks/RsGuc4SAyDUfXk2W8TAUVU7XxRbWLT7+Tn44npL+tB4z4rBjcXDcljni3wMMmrhbW8FMYYoDU8nIiAHe0iP8GW0aipTb2n4Kk6TT8QqIrTfwKMP4zo28BkayVPafTzGDpe4TMCqe3IySo7z/4HUEsDBBQAAAAIAAAAIQBMKeznMgAAADAAAAAaAAAAc3JjL2Flcm92aXNpb24vX19pbml0X18ucHlTUlLyzEtJLUjNS0nNK1FITC3KTMxRSEktSU0uyczPU0itKEgtysxNzSsp1lNSUuICAFBLAwQUAAAACAAAACEAukkc5B8DAADsCQAAFQAAAHNyYy9hZXJvdmlzaW9uL2NsaS5weZ1W30vdMBR+718R8pQrtYqTMQZ9kI2JICpjGwyRktue3mZrkyw5rbq/fiRNq15zq/OpTXq+7/xMvtZGdaQo6h57A0VBRKeVQcKlVMhRKGmTZNozG82NhWn9yyo5vSub1I5Jc2xasZ5orjg2SZJUUJOOC8lWHxNCCFE2AzkIo2RmASuoed8ioz8vzy+LT5cXX85Oi89nX2lKLBrmOLLytmIrckBoZpEjUPfat2h4e4+itHS18sQ+PkPyOdbsxGz6DiRe+S+sAlsaoV1iOT0Bo34IK5Qk53xNbMfblpxefSdaaGiFBDqy2n5N8sCd8aoqbL8eV9YRYk5L1XVcVjQlBv70wkCVfzM9zHCPGiGMVqpEZWhKGmh1Tq+MkEgGMC4QS7isfAxC1sp0vgUhDG1Ak/wZndvmZorVrfxnHhJndH/fKIU0JXivIXflTEmouV8xWnHkcwljBGi4kBODkPhA8OHwcAE38DaKOlpEWYAqCjs+GlG1wEghxiBni2e0PssnVYg0KwYErcrG7khkN0x0G/s3inp/vABbcyybuLPdIMk7iE9ftBIwiNLZT8y01P1C5V7TkGegW2V+g4mXbSF9A7b3uUy2F0rCfFY4NgQVabnFTGMIGYbINJSqe3QsYHjbOERwtyA2Ddq3QF8xEREUihb+G6QGMC3XE65uFX9c/+xod10WhiNi3opO4FKTY8H1qPvFG8n00h6MLRTW3YA7uTp+t19BPIJ30zUTwcHA20Wwm7vdXlXlSlQ2SpRg82t62yjfJeq65Q4LXSts6M2jMvqNkZGbjSU5GbixLOiKfzgfloVkg6iMamYzrTSbhWa0EPWDUU4mYRk11p8vJ8kZ3GkwwkU+yXIQX7eVzMba6RBzop5Vfacte2TFVikRsgLpLqFQlHbL+6RD2+7dOZscB5ttp2GbuTyvqVerm9RnfR1u9XnpFGVe+HvpZldAI/LlaqCzewjJL9nennMxMVt4mSZcOA9EbkaeNM8PzUgZ+uuzHl2l3t7mbHuUVq7NI1cepspHRJjbTFerJElETYrCaUBReKuicL9bRRGyH/+9kn9QSwMEFAAAAAgAAAAhAOPSpVW+AwAATQsAABwAAABzcmMvYWVyb3Zpc2lvbi9jb25maWRlbmNlLnB5vVVNj9s2EL37VwzYi9xoHTv9CLCIChRB0VOBPTS9CIJAUyObWJkkSCpeo+h/L4aiJCpWi80lAkRI1Hszw5k3I8bYx6dPD1p1N9AGLfdSnR6MlsqDww6Fl1pBa/UFHP+MDRiLjQy7LgelPXBQeIVfn8AJbXHHGNtsAl7oLvIdyIvR1sNH3SuPNocGW953nixF9K6R/KS081JM8Av34lyjtdq6zWbTYAvuimgyb3t/zpexyJPSFpsc/NmiO+uucUW23+1/ymG/O4QlPL4LS3j8gZYfaQmvP9Pyfrt93ABAYgYKcNp6bDKHPpv3t9sAlG3IQ4LXFri6ZbS7hw8FeFoO0GoLHqRKoNEXXZZLh/AX73r8jU6csT9ni5feeTgikct9DoeKRd8XfsJahhj/tiWTDauCHxv8UJpKFkCOVf8EiuAeT9reRpZIWCJhRZycmZ7bE3pHeddGO2ygSAuZddL5bX6/FciuN6GmxSiCbNgntzxxy5XSnoeasmrOTkwy353QZ0w6YfW1YTnskwwmMZY8Hrums1U7bgyqJuODz/GKMRE4yQqrKnhTwGEKz1B4idYWUZnUUYhRqqQsRP/C+ghKC7E8xL0Unibvg+23IxmkA350qPzQpCere9UMyYwaoWusWGlW82KimoYWqo+3OoBGUU2ER7AlO+oX0sQss0gbRHLsxTP6oCz/uNBCrPt2rQ8GLn0YndH32HVTNpNafwe/W8TmBg0K6cKMIfZZns5oH8IoWswqEEFY0CAdGLSCjtMkSgx2+rpK3cEfNIdAK4E0XVCB5884zDsCtvIF3W6yVOcgOu6cbCWNoxqKxSDLVvU6nrGauyvd+7IyoQ3G7zmU1Tade+W+mpzMGpikPEe3lF2oy2hkWZ8lcJZ+yAGr4JdiBt9jE12UE6wiLS7bjnaepVp2oNVXUlM5HOlVMRq0dTgkqXCQ1sid+2Z10C1jF5JG3EgZ5uQSQJImLyunE7JZgr3JoaVbkdHALJk3rMqntza8jWOJLMDDArowOB2znENU/IKUvgJCz92VwpvCm1n1U2pM0a5tq6JV99sxwCIN9B5lbGzNwht4Cxf+kh1y8AbeQGu293iLgnfdClitgNtD8Q6+hxQc38n6GmtuBG/oj95fMjEkdf71TRndfabh67L4ix9SNLPa17NUwlKvZcXEhj9sSczD/zOpG0XJIo1Vs06oe8Ypv0jHujyEVq1sUAksJil/48pepLC6/vr6Bi6PXMr4lMTtbKNDleyvWJhyW0xP/yWj4cmi760Ked78C1BLAwQUAAAACAAAACEAfhTp7H4LAAAXJgAAFgAAAHNyYy9hZXJvdmlzaW9uL2RhdGEucHm9WWmP47gR/e5fUWAQrDSjlt29BybOKECCmQUG2N1ZzB5A4hgGLVE2p2VSISkfa/i/B0VSp4/u3hz6MCOLZFWxqlj1HpsQ8k7uRCFpBlLxFRe0gF+5fqekYEBVuuZbpoGKDDb0kUHGDFMbLrg2PAW9oUUBulpqZnQ8Gv0kK5WyKayNKfV0PF5xs66WcSo341pm83L3jhqqmRn91SmBDVdKqin8UhhFi4PhqQaqrWT4QUJVaqMY3UAqxZYpzaWAVGYMuAa2Z2llWBaPCCGjUa7kBhaLvDKVYosF8E0plQEqhDTUcCn0aOS/raleF3xZ//yspajfFRWZ3NS/KlUUfBkr9q+KaVN//Y2XOS+Y01hSg7JqdT9Ss270HOimcLN+/PBdPePDhq7YaPTDX79//xMkMBsBAJCSZUwbxakgUf1FlgWrfy15ekjbnylV9eu2XWJUlT62P/pL6E5wsbobfl5Wun7dSCNR7Hz0/YdPnz5+ggTIhaBWbajGLlRjxQpGNdPjzKfVeDuJJ/GEjEajjOWQ8RXTJkBfTa2LQrj7C2ijplbzGpI6JrFe04evvwlCO7DjZm09HMuSiYCoJQmBasjdOnxyqWBZyPQRuABumAoKullmdAp5rBjNgvvJw1fwCvC/MIIlIWG72CqPqzKjhgVWitOrmKmUgHW8Zntve+i3UlKl2ULJnQ4M25sp7iKCHc9wa1yYCNaMr9bG/rDbNFVZsFnBtYkA/507/YSQT07NlhY8A0PVihl36vhKSMUyUGyFiYvyzRqWcs80pAUvS5aBkWDWDDimU4wnAIV6IVEjIYHZPILZfFT7quCCoavQ+FiXBTf4RQcdr/AchDR2Zow5WXYH8UmlMFxUrPmo5A415YWkJtjWi0KrcIvanCzUFpCIhKiimTbvKi6YCJTchfAW3vSVKso1g19pUbH3WDOCnLzflyw1LAPr8baC5ZwVmZ7CEdWeiIspPvsIDhHsIlhHoFOpWAQpNWwl1QES3MVs+k1rzn4SwWECCWzoPpjEkwj2YdT+OHTk3kdwuMeZXATOCzYjwgj28Bp2uKwZcfkRRnCA17BuhfAc9vfwNoH9BKRCeW8TOEyecP1S7tH1ztYIJdw5w/HlMOk5124ZkgSsAi5MUO8+xCAFkwju7wex9okU07JkIguWct9azAqegzWzL+ttAvcDu31i1mKOpJ694BmZDtbfwX0EZLmUezLFDZ66KjV7Mi9+EY9C7kQb22P9VieDP+DD41KXK1/FApux/ogrKU2neOGLMwQH4s1jxlVQUsWE0cnPqmIRsD3XZiEf7U+nF/uGoBsGCeSkzteHyf2f7t69//nuaPWd4t946Y4z9gSbmNLAuFk86hxSnBFbRToIMar40TeomOuFfw1wXieyJVUo2K7G2rLQVZ7zfUBQd4yjnVNTKgxP3mAGLlZwrI05kQjyotLrzi7xsRWr3z/jShW2jufk6DrMadyTY/iGycok30xslVdMl1JoFllzfQvYnbcA6ylvmy15k97Q9Q5RK3heo7Cy4p3i/V5xxYrXia1lVybyNslYBn+E4MsHr7q2wB7TcwO68Wh9N4VjR9x4DL3NnOB7/rcrYfJpdCFj0Ofhk0ftg3Ddq6Pe48cpHFFEt/zaMCpWFjT1Gdl2+dqCf/Dy2yZhMdC1uEbKH+BjyRDL2P6XMoFwBDKuWGrwsNNCS1DsM0uNBi5SuSkLZlhjoo6Hu/cqYvQlduhhv7vUejalOXS2iuYOigt+qkGDYlgagraGRGAU5WKRyko4rAAJvJlMIkQD/a8P+FUzltUfvnq4UIF8mYgV07LYMg+gZGXKqltCCML4O3PsaD/dbY+N0tOdPqKukytAPIfAyxgD2VDBczzIiJhJ2NSdTl3xqfkeRzBEjiZM4eikXKsW3mWtqszxhBghtDOlVg4JHJt1BG0lU+ueqP2aFghMNZmChdmdEVthceB46nytKc/we6mkkaksyBTI3z9+97EBaXLDjW8aDpb9GdiWFpUlGn5AQ6rkrgFxyGcMyDznKe/SrQ1DSK5jj8JPDVKzpkZgw2Lbc0Bs1LBQttELIwjIlhakkzlhJyJ+awtE0Vjxe63NdTX0d+OBNia1w2ft4Hxml80xBgRPC5n2FNgTFAFxIJ5Ma+DfnRO6HdrsQuy6yLiKoKBLVuArJJ0ssBM0gXHtjHbILmiHzmU+tyXj02h/ySLFUqkyHXUpZoMkFn60AeANCL9a8XpOulj5mv2hWC2VYVkgbK4ITJDzMobnl4zdkjHBOcLyCxEXcsdUEMZMZBqtCUj8uVyRsN+qfH2cwFufhm9dV3MSL/THC5Xyk2v+LIOjqzDgYY7fSgR0S3lBlwWDY0d4t3Hgk66lZqLduGPq8Sf7X4AFIIw1xVrvJfijM9gSesvlCM8i2LDNkiEUBiaqDVPIA52iCIZIeIDfsPwGbn1oXX6OBly1ajJyCOGGcx22WCwPhuk6Gxw08WrOcYRNJHuj4OBR0zWt0svwwTKTmqbW9sWa/3ZuVZNw8ZYpnh98V+k+VIiF92Li3dm0+Db1IiDjzjEZkzBWng7GBF09m8zhNZDY7I0r991H0d0C+Sok0HNLqzuMM4b3QgGpTH735k7z1SB9rrDjDp2vtUR9D52LcaXHnuzzBJHKq7Ek2ym8HIeWjELiZ84c6TkX26qtGdTFKdYCcvTCvuhwrC/mJzgGlozCGB5CGPs7i/hNjiOWi9Yj/gbDDe0GU9f9CefRwufcafh0MuDJbbQ9/tJDLGvEatGRGcJruI+eWOePPnLOugrcXPHqlfPmE3KpYpRMYQevYP2UCdriAjKFyfWZbZO87dagbZxjV5LqEhMOiB2erDD0NQbz/LLvyT8FiT9LLpxkHUbARCqR9CXucJ1H/Nws3/1uRvl6hMkz40NwrwuPQ+p935hu0xijZM/39XkuvcnU5/mNmUOgY+vv5fnnEb1AC/vgob0tuZS2QCzyxE9u2ekSdvOAt4PcvPBmbipT2QPVdmdc5HIAh913B8mmDfzpD3eOI2LDDjLqz/OliVtRs6OPeAQ1qhQn16kjB23a7mwBfThvxbV+bZlKTuqrFNxcTVc6qY9f4qzalDrAGReSPLwptr4Vuyl5EM2bSmri9KNjilkNkvASE69EvYgWNB2Hxe9U9x2XFec8y5EGvGe3+PGoZnX6zq2vLQy6kDiedcydo3kOVByC7mJsdF3ZN4UhVZl3kNUZYiQfMiYMT2nh9mqvPJkwDoZroKmSWjuNFszaCwjrBO/PVIqcr/o8EUE1mXr+EFO9KKXm+6BzVv02p3WKjz3Zaieg6e2w5VztIOYt5nLGUxMM09WrcR70TOwW17VcqpNUOBJrmrMFZlbgNhhZJLx4ZAedfEsLza4m2A0Kfzlz61kRcIHRSB6uyvbMHS3uX3Ys2rj07j16dxtff/XmibsNQuoz0Qk00EIKNgW6lZ27p/pKyAYOf3iw6CqJyJliIm3/VvKSW5NW9d32ePGipAH55/zVpkp/1nN55jX27hn/Ffb+O5np72Sl/zdG+iI2el5XWiLaSSR/u6IxaJpnzV+/YSkrkXWvQ54mji8mpucc8wJlvM0zfwdvfClnfAFfvMEVqRD/G4J4m9QN+eJFoniT8T2T0T2X29j+cBVQ8ix5GampMzJ5Gi0/zWSQxCR9Jjp7mMOrATudfTm/YZBjN8kVbnOOeftfniIO1n3d7ULDARJ7FtwNpI1n0otq4sE8ONySdBF734bBHdhlTO4MOfO+Q1+JX+QFe4TdDz0C7KQLr91RSM5wdScZkouAusXSyazxTwTWJ5YJNlAab66uo+kzsLClxX8KoAfS/qu4+XmwpnFTR4P1ElbmxP7hwF0h6+ToIZ5XfMIKfZH8DnDmLSjnJl00aIDqrFHYZpNzlIotK+khTxtMnVyGmzXgxGe4gW6sPXhzNo7+DVBLAwQUAAAACAAAACEAFwcUobYGAADDEQAAHQAAAHNyYy9hZXJvdmlzaW9uL2RpYWdub3N0aWNzLnB5nVdLj+M2Er77VxSYw0gzstrp7OzBsxpgN4sEAwSZxiY5CYZBSyWbaYoUSPqFhv97UCRlSXb37iY+tFrFYj0/fkUxxr52aLgTajvvtFAO0BhtwOBB4PET1MI6oSoHjdEtfP/1+6/wzyfAA5d77oRWOWNsNvOL63Wzd3uD6zWIttPGAVdKO69mZ7Mo+91qFfQrLSVWfrXfUGPD99LVonJBp+NuJ8WmX3/ibjebzWpsQB/QSN4lPINNBjW6YGrNDfLiBy4tpssZAMApg3MGxwx2UAAPIpKRkKQk3nixUA6NDXaggJafkkUGrVDJCT6QhZN/HlOY+8UTSdIU3o9Vz/ABdmSenrte9UySNPVualS6FYo7baCAI7yHHYjmJgVAaTEukk/6ZwfzSYzemkG3N2oa+8PEh7c9vH6GRTC+yBexli131W7tG28Tx80Wnc2gM0iNoP5kILZKG6wzqLRqRI2qwmKRP37MQOh9scg/xmozxn6ptMG5NjWGDZJbO+dHbhC2BrE+B39CbaHRBg7C7rmEWvCt0taJyoJW8uyBRRb3yqtjDQVYdInhaouJRNVHmsa6ekeiEV6zXHkZORjyAKHAauOwTibJPeO5kLzd1By6JXQls5QCW2V0DKisxa9m3+PJI6UZWb2qwz9G1RmUfWxaOaH2eBVWuu24ExuJFKzwgQqKb0hXNBAzLMWqZBV3uNXmvBY1W0FRTCKYLIbU6bdB6yKSB4eTfMXyepLG9jYbfaL8xwEEWZr1p7T4WStMxzXx7oQFpR3QInBV/1/maePgAT4XhKppBa91yQ22+oAJ7Rm80+9ZKGo9c11ADv1Qiga4Oif/LYyNPt1RiO84RbLIP/rubPSJ+hMPwggMY9dxdezf4uuqzSjKAbo57zpUdUIxjmLN/LaC/vRo13vlLBTwQsIl2H2bdCWjlwAP78fjn8IePKRe6FeFgpKqlflosiH81WXko2SNIpNAZ+7ahnTMPkExG3nJoBywM6A7nr7ByipSUJg3id1vLLqlJ3rimrbjRlitBskN++CJt51EW/w9dsSPjacvP/Uj40vLt5iFx78NP868ljN7R8RP0yiXmtc2ib7hAdiBy7zSlc5pmaW5QV6vHZ5cEosf60TVNyUTZNufvCWYkm30CS0LSRtK+m0n0c5rfkIHYg2hGI/GRIoe+uSDkw+fUMlGE5etBtz1reDjYFc91niwpXiLHlBVyUIyVclIGHOpRn4i3whKNERqkMrt91/dss5opyst2RLYj4H7LW9x7nECX/Rvn2l0AHfQiBPWo/Z+8iTS3zf6gcGywfagy5ajjSONVtdo2RJeLkEYAtV71+2JFQd0UTv89JsHGLKRYt4+18IkeBLWrfVz4IVr7clFOEbHnZZIZ8gJSSdoKP5o1LzZx+ususXLJMiGvZDDSz4y+Rp2poQzUi67VwHQDUEYrLSp7TBB+8j8thEC/PsEZANjTImgFdb6szK5Zky2jTHqDQcEriaXkJuV/uxs0SWjlQxKmlEDICaOpgMjJjvhXG8qg/fv71Mp7rMqwiPSgk9DOy4DL+M50LIpn/E84oPoNhIxnu95uFGvsPFgvWSdwUpYoQMx91LXsRU8+HH/bTaVfri+Nh1b3YZbMoMVl/IvWFNja5EEynj0ViU9VxNO8Ccz7GbLaCabrkZoLaF8eV7CIZQpg4OvXS4ctjZJ6b7x7EnCV29oDVUudIWtLndFXw2+hqp+A7/wFnuI1xZ4ZbS1/nTbT9AIY9110gBdZKudtqhgc47EdeBS1J53wV98hdrmk+MzjqFc9rZuTk8IoAjDKtcdqtG8CFV5OHDJ4IHmTCMkrgM/p3ml1QGNS9h/fvwXm6K8NvzYG6UJmNOfgPOpoj/mDlsf6qSkN3HefVbRrv46dadZaem/dF4IO0tgUrSeKBv/RgAfYX0JbGv4mV3KYDPcZ+6NUlK5wcpxtZWY0LfYOYP4nea/vdKMGFwKhYWPIIOjqN2ueExfN+a5826JfmScTsHjgkzP4dvHNM2IjP3EDIG+G93A361WF3gJYv9p8G61zB+bC8ugEVKGcO483fciMGLsRg/p/9GJsOfNXvypsrEzSqmP7I3K3dpaZLDIAoZzvyGDx0Uac2YbyavnV5D5Rt2Tv2XwXTolBl+XOAKX8OuT/5ykufjDE9Bt7KEfNSFueLje0whSn/xMKF6GyXAZXSeu9n2wx51weLM6DT2kafkBk3ih8OPZlO+o/8vFd5d5nNW/d9uY9qAZbhvX2X00wmEY3n721/u2s0kk0wyEqlG54jFOmXjnjsuzPwBQSwMEFAAAAAgAAAAhALn/jJDiCwAAdSMAABwAAABzcmMvYWVyb3Zpc2lvbi9leHBlcmltZW50LnB5xRprb+M28nt+xRwXC8hbR3GyTe/qVgUW3e21QNtd9ImeEQiMNLbZSKRKUo69ufz3w5CURPmR7V0/nLBrSeTMcDhvjsIY+7HmVQX/fPczWM2FFHIFXJZQqLrhmt9WCF++/fLtubG7CgE3vGq5FUqmjLGzs6VWNeT5srWtxjwHUTdKW+BSKuvAzNlZGPO3StymNVpecsu7md+Nkt1zU3G7VLru3k1722hVoDHdiBU1+nUbbteVuO0Wfcftul9tx+vq7OysxCWg3AitZI3SJhM4/wJKUdj5GQB0mFbpYn3mRqze+Sm6ClXXwkIWcZEWayzuctXaprVJD0nXgq2EZVNgGjfnDdcG6eXrN69es5spWNza7Cfd4hSMLVHrLCL6+s0v3//87bc9uUlqrBZNMnEjuC2wsZC8/fGN1kpPY3a+5FWF5Tv/5qYnR/hnrbyT6l4yN6XRtlrCQw/Hmp1dK8nmvfhTP5JvUBuhZDKZRsABZgQeHkaAfnk2D3zEJHhxx1do2Dzigq5mfsRO0o6Lxsuju5ZKQwNCQsLaympe7awoDAndKbR/2AjCplfZ1s2OHppdoQpllaoMG4g+xsy3Jc/5houKfIDNvZGkNJwKM8yMdrxq2jHkCm1e4kYUmEteYzKbgFieJgVYGYTvlcQ9TnqqQRQO2cM8BjN33ustkoQ2d/7gQbBRxdrMQcigBFGvzPvo/ZbbYh29E69zMFb7V7+DaMAglg4cMvj4yo/dK32H2q8CGcz8qEbTBlrwb7czyMIGnS8Sk0d80amXfDxSbAfx29tv33pn9V4ImaOSMN1KwyZw4dj3NJcBJsWtMNYkExfbpLIdY72YNRcG4StR4RsH6nwpWbIfWgm80sjLHXgic3jwRB8/g2KtlEHgUtk1arcwKA3n5558sK1CyaVYQeYizxA0SE2ZsTpxRq7RqGqDySQyKK+4zN+GYae+zP0Og06HmfsdBr3mMn8bhkl/Gf0MQ0F9WbgPE7UqscrYTlXq8lKmjWXDnGqsqMV71Bl7VfL612iq0rNsls5ml9ESKFZr8oeC79zc7HqYlLdmn3leNz5iDgGNF2vMvuKViQZLtKhrIYWxothDaLgVKAs8EGFRKYN5rQwXRTaLd+tH0og1U/AKs1n6chhaVqKp9Bis0ep3LGzm7TDaGa8xo59Ir2RIubrb30pTKWv2tmD4BvMGtVBlFgmz5tu8RJu9nAXuvak9GwQJpTAUVpzjVKIQFlaalwKlBV4Ubd1WLkXPwa4RNP7RorFYekMCYQC3Hi11lCkaQxZnjSixsvk4zUbhqyfsEgE5QjRpLNcWy7y1BYU4USNlviU9JOz5b+fP6/Pn5U/Pv54//27+/Md/samHWdUOonOVR/cbB4G0viuFTno5k0i9gBqUJdU4B0FjyR5ISY9D2qG6hMVI6b0WFnNK5QlNpmVbNyYhhCkIWaK02dVkCigLRfAZa+3y/B8hCjwV+Ic4FIFoNGjzBvldXmOt9C43llMQ8zGYRAeZF0iDepkXqpUWdZh3fguZC5eJD0cUmmJH7vk6CIZo2oqIOyKpTy0eKNQw48CyHz1C3VKZD9F88eLhbg4bl8rvprChbO6NJBUWawrYYgl38LcMmMNjj5Mjod/TTp2rlEIHAaDlC2awULI07Oa4qODcC/K/0JCn69RCGtIbLPPbncWwxkCAXDQorgMMukkC+xfAcEu+TV7j7W3y14ys0UKO8B7IwplLwGHVyRR6qcz3pfQ4mcKyas06cplQL3rsUG344wEerzd8oDfx0FBxQAaffBxilhUVHgyqDeqKN3NYVoq7UiK9OqhDqKidhSBbiVrYvbrDM+s5GPn6hedcOFl72BBKOwp9RKXDTj6a3KtgOjczc7BtU+GCCiRI05QMIWH3a1W5MwBts6TVjh8/CtXsRu+bq/jVFa3ADcjmiUIpqmhTeurA6PB2Goq2GEPS+4eLrx4gJc13U6VYobHRJG27R2w00sbPDiQL2fhV6U4hnU9SuebETHNc7hJ6caOu9t8XswsmDkRIjzfZL/N+4VWLvsJj3znKdWss3CI4YlQkXijtzLMcomTH4edwOSNWRmx/3pvRE2v9gH+0QmNP6YuMSFFJOqL1RdZBsIkXmGmwgMydalPDl5hXipdd3chLHy0OIkJwX6X6QEl0FowOzuzGz1rdOg9xIYOomiRxGBfANrxyVtJFpg8uJWo61VEUJKoL5t/ZzWLufPSGpOie/EFnD8qTKAn/QS+YKNmN06UmRXogn+n38CALs/Fk1IJwEAvuaHGidRRELIEHmrlbmhYtjefqmfMOKLS6L2HNZVlRDSEMcNmXScCbRqutqB1JsAp+Eea1poAhVlJpLEHjilbzBVU3eFr2AeJPi1/i1uaCKNZ8myQ8kuHJfU8oly95W9lsNoGP4NI7MIld3TsZeCYGs6Z6Qd2PRBV8UZRmAKOrUNIK2frDWEf3Vm0J2NG4VVvS4BiLgApucaX0LmI7DIlDhFNqT3lDldu4TdNd47PY/iXKLEhzehomCCDbE8dpjG5XhNQ9eyWdxrm9VdvsVm1PQ3CNnCAWVzfwgsS7ePkEPWGcFcdnifgat1iOj3R29lEWzGVFAYY8JNQ3K+syg3HRndJbWiI29JA4PfVQhUZu8RtZ4vZY2UpFSygmgomH47wv76kCkd2BaThUxZWLyz3/w6klrJqbNb+6/oTNQ3bruYlAay7FEo09gO1cuQMIfhz3wbSyqlAVmwOLuqyv3k1DqBmHFHKMvVDymfM9tVyKQvBqiDimUFrIVXQMZQatFXJFRd+h7R9tKdBFOTCjn/FwKNKycB9PUgW/f/h3i6g2m6V/Hw+eaFHEZ9xwH0/GGTOLXwawWNSde9L2jaJDZyLKsR6pEKCOZGgD+lxzUExE+V3dG3cgJalOwZ0pKHktbqbhfw/6DH7luoa2AeTF2hNUkrIHL2zrajC+wimZtxElBqI+TzguhDbOlzZXqagpETjf6CzMp7+LDa8YXIRkuJjdLNhSVL7zyG6CAzk9LANBYVwlO46mQyfse2W/Uq0sffVygmzU/3AVnqvOqqlfYeq2OvXmRbKqaJPBZsCbCJBJBDOY7ut8IF9QMC/zvsSYjZKKxkLpcigUxltyY39WfJ7UKdkF+XmKR+X3lAyP0d4nfbo7TIWil5M7EBfUbT5MhAO62clirZUU7zFE1+76UPNgT6cUaaYgIeuVfLBs0Hqw47+o9RH1/4uAgld3FUTyROtgAi/gcjabjQn4aNDhy73JkS1/lEGFMolkPYmOAGPfVPcmxa0lmk4PQw0SLMsVE/DiRTMZvo/ElPeFSSt7XifwHK6uIctgdig031dYsgfS7OMcHiK8xwv35nczeQweOKVdZg9hF4/ssK8w7oR0tNOI26dbIiSMk42QsD2CGe+G2sWeOmRUglDd/QMGaj3kuHd1gBfVOofzUe3z0B1S5r1U4jp2fqy2ncKoip3D4ubx1EqH9ZNj3n+lVdQl6U71ycpOI8wpMCos2RGklL771iYV9eobdxiL0uVp6JpvX6NPf5dTuJxNRwn65ghieIo9b5jsG9Tx9DP4aY3kdLLkugTT1jXXO1hzXRb+FO+ZyC5nM2f9r9591tWAqtX9SS0i2GG8nM3AVKKghoU0FnkJaumt3n0Rh1qYCrlrHZ9fwho5HQFxyNCNxsJ9Zwz9jGGPC9bPsRt/nPdaXEKNXOYbXoky4VGTgi43ChnwBadgMBskGLXkXIcscaAp0Uomrl/qB4x4j8MHxWFhL5BFqHduFnSnE/JhXOdNFnHY72Ixn4L/N5vC1U1UQ3mk69lxtNnTaLmhv0T4wIqXR1A1FryqDtH31eDhqBtxkhRv6FNLXlTcmGz8aZquYsF80qa+6Uk277oddm3tgoIwyrZGTQZ9xOfHoST6Dk0XeYEsdnlzPctrk3mlyyatsRRcJiFVxd8NR1ifXo+wGtQFSsrJHeYUPr3ex3bbcw28eD3ex/w98CGwZJQKfGQ+LMH8bEgUx+ZzbvMh/ueUQUa5Mqrtj6QQ+ouD0Nv9QOZwLvAn2ul0HbTUjzrQiNiJDrpHPPsPUEsDBBQAAAAIAAAAIQBG7kRrnAMAAMIIAAAXAAAAc3JjL2Flcm92aXNpb24vdGlsZXMucHmNVU3PozYQvudXjDiZfR1Edm9o2dvuseqh6gWhyOAJWAWb2iaBt+p/r2zzEbJv1UZKwDOPZ575TBRF36ehE7WwYEWH0KDq0eoZmORQd8yYM3swjcBHB2MWwYzDoNEYoWQSRdHpdNOqh+v1NtpR4/UKoh+UtsCkVJZZoaQ5nU4cb2As09aQDmVj2wyEtBSMeMflVd1Rd2zI4NYpZmM4f4NOGFsIacvsBAAgbhAuw9ccUlDaX98OUllI3WmxBF/hEi66j2bCIPzOuhG/a600iX5VRlhxR+CiR+kCMj7uVxug8c9RaORR7K11zFjIoWcTSenK6Oy5BICxOCyACwWtRsmJZ/oJyAXOq+04DnCNdtQSjNIWOTFoSfFJM9mgN++cvcGFeqtxEJRxvOT0ISRXD0Megm85bVE0rf3/Cbbj0OGS4oVLsaWNTBRmCr2QZII3b46C9xYH6bxJg98lKPe5KQ0zCLlWPgACp43OET49wb2XD9HlEv2gkYvaEi/sFceOhkbpWYN0k2ZgrF41jXkP+fBn1/VPx2OKgqxW8nYQCDUezhzvon520rPpyjFUAHL4kqb0FC8tHIbDKl23XuCnxx/vws+UGsyKqpitW+RX2ZuTB7cUHpCH8BLTsgGL7HMZWGp3MYfCNU1K4UGhjUs3Mi4DkOcQPVrVYQTYGdz7hjqjLgsvKa7UhIaCqZV2T78L0Dso6fI9bUVLKcwphelCYb64Cno2T7OHZuz8zLgaJYeyrR8fVDGn2XyhMKXZdCnpC6Ax77n/PSpcgXL384JXYy7UeBSGWuXhcVQtVcuX51F5R10pg/kP1pmne3GRhjT4lEG+RJr47G0KcYMKhIFflES3qDqUpIpdTdLsNRIr5IibcJqnGXKoEveScLSsbkmc1J2SSPbJcdoio1CkFD6XJbzlMKU/ay8UvgTtvGs91YQNA0pOHHQ3G4q/6qrE5XgjscOW3njCdWbn2inZrGhx80vau3zujrBzSr9Mie+d/2hCPzBJzSzxkJg+SQL6IFoubiS2kXCNz6Odyh/oN/fT4JGPOVDXXnGRLc0SmsBN1k+21iTqxi14wtHUKLmQTf6bHvHVhFaPMGTbcFVqWrxTcH/AjdJ+p76LYR8gT7JwHsukHkYSJ1a53U7ilfm/KJdwPtR668ve8n10HPPced2LqB5bB/wVrUSvgkfZRptCVFVqijIodltwDobdy5yWFCJPOMoC8b8P/5HOy6FL/gFQSwECFAAUAAAACAAAACEAeLVKHWABAABRAgAADgAAAAAAAAAAAAAAgAEAAAAAcHlwcm9qZWN0LnRvbWxQSwECFAAUAAAACAAAACEA68zDpyEMAABvGgAACQAAAAAAAAAAAAAAgAGMAQAAUkVBRE1FLm1kUEsBAhQAFAAAAAgAAAAhAEwp7OcyAAAAMAAAABoAAAAAAAAAAAAAAIAB1A0AAHNyYy9hZXJvdmlzaW9uL19faW5pdF9fLnB5UEsBAhQAFAAAAAgAAAAhALpJHOQfAwAA7AkAABUAAAAAAAAAAAAAAIABPg4AAHNyYy9hZXJvdmlzaW9uL2NsaS5weVBLAQIUABQAAAAIAAAAIQDj0qVVvgMAAE0LAAAcAAAAAAAAAAAAAACAAZARAABzcmMvYWVyb3Zpc2lvbi9jb25maWRlbmNlLnB5UEsBAhQAFAAAAAgAAAAhAH4U6ex+CwAAFyYAABYAAAAAAAAAAAAAAIABiBUAAHNyYy9hZXJvdmlzaW9uL2RhdGEucHlQSwECFAAUAAAACAAAACEAFwcUobYGAADDEQAAHQAAAAAAAAAAAAAAgAE6IQAAc3JjL2Flcm92aXNpb24vZGlhZ25vc3RpY3MucHlQSwECFAAUAAAACAAAACEAuf+MkOILAAB1IwAAHAAAAAAAAAAAAAAAgAErKAAAc3JjL2Flcm92aXNpb24vZXhwZXJpbWVudC5weVBLAQIUABQAAAAIAAAAIQBG7kRrnAMAAMIIAAAXAAAAAAAAAAAAAACAAUc0AABzcmMvYWVyb3Zpc2lvbi90aWxlcy5weVBLBQYAAAAACQAJAGYCAAAYOAAAAAA=')
assert hashlib.sha256(bundle).hexdigest() == '280db12e79b5658e7d751dbda9f9b0df9a90d1ed8c1b3af09c0327c8236fe483'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.namelist():
        assert (PROJECT / member).resolve().is_relative_to(PROJECT)
    archive.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
# Editable-install path hooks are not reloaded by an already-running kernel.
sys.path.insert(0, str(PROJECT / 'src'))
# Save the source that produced the run alongside its outputs.
(PROJECT / 'source-280db12e79b5.zip').write_bytes(bundle)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator, then rerun this cell.'
print('GPU:', torch.cuda.get_device_name(0))
EXPECTED = '9ea4e947efe74f75703ab2c5a32bf5a6af70124945a6ee9bd366c7cd83172840'
BEST = PROJECT / 'full-model-best.pt'
original = None
INPUT = pathlib.Path('/kaggle/input')
for archive_path in INPUT.rglob('*.zip'):
    if not zipfile.is_zipfile(archive_path):
        continue
    with zipfile.ZipFile(archive_path) as archive:
        candidates = [n for n in archive.namelist() if n.endswith('/weights/best.pt')]
        for name in candidates:
            payload = archive.read(name)
            if hashlib.sha256(payload).hexdigest() == EXPECTED:
                BEST.write_bytes(payload)
                original = json.loads(archive.read('full-data-manifest.json'))
                break
    if original is not None:
        break
# Kaggle may unpack uploaded ZIPs automatically.
if original is None:
    for checkpoint in INPUT.rglob('best.pt'):
        payload = checkpoint.read_bytes()
        if hashlib.sha256(payload).hexdigest() == EXPECTED:
            BEST.write_bytes(payload)
            for manifest in INPUT.rglob('full-data-manifest.json'):
                candidate = json.loads(manifest.read_text())
                if len(candidate.get('splits', {}).get('val', [])) == 548:
                    original = candidate
                    break
            break
assert original is not None, 'Attach your full-data results ZIP as a private Dataset input.'
print('Verified full-data checkpoint.')


In [ ]:
from aerovision.data import prepare_validation
DATA = prepare_validation(PROJECT / 'data', count=548)
current = json.loads((DATA.parent / 'manifest.json').read_text())
assert current['splits']['val'] == original['splits']['val'], 'Validation images differ from the original run.'
print('Verified all 548 validation images. Training data was not downloaded.')

def run_evaluation(label, *args):
    command = [sys.executable, '-u', '-m', 'aerovision.cli', *map(str, args)]
    log_path = PROJECT / (label + '.log')
    print(label + ': starting', flush=True)
    with log_path.open('w') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                log.write(line)
                if 'images, cap=' in line:
                    print(line.strip(), flush=True)
            code = process.wait()
    if code:
        print('\n'.join(log_path.read_text(errors='replace').splitlines()[-20:]))
        raise RuntimeError(label + ' failed; see the saved log.')
    print(label + ': complete', flush=True)


In [ ]:
# Same checkpoint and scoring protocol in all four conditions.
conditions = [('whole-640', 'whole', 640), ('whole-960', 'whole', 960),
              ('whole-1280', 'whole', 1280), ('tiled-640', 'tiled', 640)]
rows = []
DESTINATIONS = []
for label, mode, size in conditions:
    destination = PROJECT / 'runs' / label
    run_evaluation(label, 'compare', '--data', DATA, '--weights', BEST,
        '--mode', mode, '--imgsz', size, '--tile', 640, '--overlap', 0.2,
        '--max-det', 1000, '--eval-max-det', 1000,
        '--output', destination, '--device', '0')
    result = json.loads((destination / 'comparison.json').read_text())
    metrics = result['modes'][mode]
    rows.append(dict(condition=label, AP=metrics['ap'], AP50=metrics['ap50'],
        AP_small=metrics['ap_small'], AR_small=metrics['recall_small'],
        p50_ms=metrics['latency_p50_ms'], p95_ms=metrics['latency_p95_ms']))
    DESTINATIONS.append(destination)
    print(f"{label}: AP={metrics['ap']:.2%}; small AP={metrics['ap_small']:.2%}; p50={metrics['latency_p50_ms']:.1f} ms")
with (PROJECT / 'summary.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)


In [ ]:
export = pathlib.Path('/kaggle/working/aerovision-resolution-results.zip')
with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(PROJECT / 'summary.csv', 'summary.csv')
    archive.write(DATA.parent / 'manifest.json', 'validation-manifest.json')
    for destination in DESTINATIONS:
        archive.write(destination / 'comparison.json', destination.name + '/comparison.json')
    for log in PROJECT.glob('*.log'):
        archive.write(log, 'logs/' + log.name)
print('Download this file from the Kaggle output/files panel:', export)
print('Prediction JSON files remain under /kaggle/working/aerovision-lab/runs if needed.')


Send back `aerovision-resolution-results.zip`. No model training or final test-dev
evaluation was performed. Compare latency within this Kaggle run because GPU and
CPU conditions may differ from Colab. Higher inference resolution is an experiment,
not a guaranteed improvement for a checkpoint trained at 640 pixels.
